In [2]:
# 0. 라이브러리 세팅(feat.현지 튜터님)
def load_libraries():
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import scipy.stats as stats
    import math
    import platform

    pd.set_option('display.float_format', "{:.2f}".format)

    if platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'AppleGothic'
    elif platform.system() == 'Windows':
        plt.rcParams['font.family'] = 'Malgun Gothic'
    else:
        plt.rcParams['font.family'] = 'NanumGothic'

    return pd, np, plt


# 1. 데이터 로드
def load_data(pd):
    df_port = pd.read_csv("../dataset/portfolio.csv")
    df_prof = pd.read_csv("../dataset/profile.csv")
    df_tran = pd.read_csv("../dataset/transcript.csv")
    df_menu = pd.read_csv("../dataset/starbucks_menu_260112.csv")

    print("프로모션 제공 데이터 크기:", df_port.shape)
    print("고객정보 데이터 크기:", df_prof.shape)
    print("제공 프로모션 데이터 크기:", df_tran.shape)
    print("메뉴 정보 데이터 크기:", df_menu.shape)

    return df_port, df_prof, df_tran, df_menu


# 2. Portfolio 전처리
def preprocess_portfolio(df_port, np):
    
    # One-hot encoding
    df_port['ch_web'] = df_port['channels'].apply(lambda x: 1 if 'web' in x else 0)
    df_port['ch_email'] = df_port['channels'].apply(lambda x: 1 if 'email' in x else 0)
    df_port['ch_mobile'] = df_port['channels'].apply(lambda x: 1 if 'mobile' in x else 0)
    df_port['ch_social'] = df_port['channels'].apply(lambda x: 1 if 'social' in x else 0)

    df_port['channel_count'] = (
        df_port['ch_web'] +
        df_port['ch_email'] +
        df_port['ch_mobile'] +
        df_port['ch_social']
    )

    df_port = df_port.drop(columns=['channels', 'Unnamed: 0'])

    # 파생 변수
    df_port['reward_ratio'] = np.where(
        df_port['difficulty'] == 0,
        df_port['reward'],
        df_port['reward'] / df_port['difficulty']
    )

    df_port['offer_strength'] = df_port['reward'] - df_port['difficulty']

    df_port = df_port.rename(columns={'id': 'offer_id'})

    return df_port


# 3. Transcript 전처리
def preprocess_transcript(df_tran, pd):
    import ast

    df_tran['value'] = df_tran['value'].apply(ast.literal_eval)

    df_tran['offer_id'] = df_tran['value'].apply(
        lambda x: x.get('offer id') or x.get('offer_id')
    )
    df_tran['amount'] = df_tran['value'].apply(
        lambda x: x.get('amount')
    )

    df_tran = df_tran.drop(columns=['value', 'Unnamed: 0'])
    df_tran = df_tran.rename(columns={'person': 'customer_id'})

    return df_tran


# 4. Profile 전처리
def preprocess_profile(df_prof, pd, np):

    # 이상치 처리
    df_prof['age'] = df_prof['age'].replace(118, np.nan)

    # 결측 처리
    df_prof['gender'] = df_prof['gender'].fillna('Unknown')
    df_prof['income'] = df_prof['income'].fillna(0)

    # 결측 그룹 flag
    df_prof['is_profile_missing'] = np.where(
        (df_prof['gender'] == 'Unknown') &
        (df_prof['income'] == 0) &
        (df_prof['age'].isna()),
        1, 0
    )

    # 날짜 변환
    df_prof['became_member_on'] = pd.to_datetime(
        df_prof['became_member_on'], format='%Y%m%d'
    )

    # 기준 날짜
    reference_date = df_prof['became_member_on'].max()

    df_prof['membership_days'] = (
        reference_date - df_prof['became_member_on']
    ).dt.days

    # 나이 그룹
    def age_group(age):
        if pd.isna(age):
            return 'Unknown'
        elif age < 20:
            return '20대 미만'
        elif age < 30:
            return '20대'
        elif age < 40:
            return '30대'
        elif age < 50:
            return '40대'
        elif age < 60:
            return '50대'
        else:
            return '60대 이상'

    df_prof['age_group'] = df_prof['age'].apply(age_group)

    # 소득 그룹
    def income_group(income):
        if income == 0:
            return 'Unknown'
        elif income < 50000:
            return '5만 미만'
        elif income < 75000:
            return '5-7.5만'
        elif income < 100000:
            return '7.5-10만'
        else:
            return '10만 이상'

    df_prof['income_group'] = df_prof['income'].apply(income_group)

    # 컬럼 정리
    df_prof = df_prof.rename(columns={'id': 'customer_id'})

    return df_prof


# 5. 병합(tran+prof+port)
def merge_data(df_tran, df_port, df_prof):
    
    # dtype 통일
    df_port['offer_id'] = df_port['offer_id'].astype(str)
    df_tran['offer_id'] = df_tran['offer_id'].astype(str)

    df_prof['customer_id'] = df_prof['customer_id'].astype(str)
    df_tran['customer_id'] = df_tran['customer_id'].astype(str)

    # merge
    df = df_tran.merge(df_port, on='offer_id', how='left')
    df = df.merge(df_prof, on='customer_id', how='left')

    print("Before merge:", df_tran.shape)
    print("After merge:", df.shape)

    return df


# 6. Save_Cleaned_DATA_Frame
def save_data(df):
    df.to_csv("Final.csv", index=False)
    print("✅ Final.csv 저장 완료")


# 7. Run Pipeline
def run_pipeline():
    pd, np, plt = load_libraries()

    df_port, df_prof, df_tran, df_menu = load_data(pd)

    df_port = preprocess_portfolio(df_port, np)
    df_tran = preprocess_transcript(df_tran, pd)
    df_prof = preprocess_profile(df_prof, pd, np)

    df = merge_data(df_tran, df_port, df_prof)

    save_data(df)

    return df


# 실행
df = run_pipeline()
df

프로모션 제공 데이터 크기: (10, 7)
고객정보 데이터 크기: (17000, 6)
제공 프로모션 데이터 크기: (306534, 5)
메뉴 정보 데이터 크기: (195, 13)
Before merge: (306534, 5)
After merge: (306534, 25)
✅ Final.csv 저장 완료


,customer_id,event,time,offer_id,amount,reward,difficulty,duration,offer_type,ch_web,...,offer_strength,Unnamed: 0,gender,age,became_member_on,income,is_profile_missing,membership_days,age_group,income_group
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,5.00,5.00,7.00,bogo,1.00,...,0.00,3,F,75.00,2017-05-09,100000.00,0,443,60대 이상,10만 이상
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,5.00,20.00,10.00,discount,1.00,...,-15.00,4,Unknown,NaN,2017-08-04,0.00,1,356,Unknown,Unknown
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,2.00,10.00,7.00,discount,1.00,...,-8.00,5,M,68.00,2018-04-26,70000.00,0,91,60대 이상,5-7.5만
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,2.00,10.00,10.00,discount,1.00,...,-8.00,6,Unknown,NaN,2017-09-25,0.00,1,304,Unknown,Unknown
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,10.00,10.00,5.00,bogo,1.00,...,0.00,7,Unknown,NaN,2017-10-02,0.00,1,297,Unknown,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306529,b3a1272bc9904337b331bf348c3e8c17,transaction,714,NaN,1.59,NaN,NaN,NaN,NaN,NaN,...,NaN,16959,M,66.00,2018-01-01,47000.00,0,206,60대 이상,5만 미만
306530,68213b08d99a4ae1b0dcb72aebd9aa35,transaction,714,NaN,9.53,NaN,NaN,NaN,NaN,NaN,...,NaN,16964,M,52.00,2018-04-08,62000.00,0,109,50대,5-7.5만
306531,a00058cf10334a308c68e7631c529907,transaction,714,NaN,3.61,NaN,NaN,NaN,NaN,NaN,...,NaN,16979,F,63.00,2013-09-22,52000.00,0,1768,60대 이상,5-7.5만
306532,76ddbd6576844afe811f1a3c0fbb5bec,transaction,714,NaN,3.53,NaN,NaN,NaN,NaN,NaN,...,NaN,16987,M,57.00,2016-07-09,40000.00,0,747,50대,5만 미만
